# Week 6 exercise — fine-tuning ("The Price Is Right")

Predict a product's price from its description. We curate the Amazon-Reviews data
(`curate.py`, reusing the course `pricer` package), measure a **zero-shot** baseline, then
**fine-tune** a model on the task.

> The course fine-tuned `gpt-4o-mini` via OpenAI's API, but OpenAI **deprecated/wound down
> API fine-tuning** (403 `training_not_available`). Gemini tuning needs Vertex/GCP, not just
> an API key. So we fine-tune a **local open-source** model (`distilgpt2` + LoRA on CPU) —
> Week 7's technique, brought forward. Run `curate.py ALL data_small.pkl 5000` first.

In [1]:
import sys, pickle, re, random
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
openai = OpenAI()
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week6/pricer/items.py").exists())
sys.path.insert(0, str(REPO / "week6"))
from pricer.items import Item  # noqa: E402
HERE = REPO / "community-contributions/NicholasDean/week6"

In [2]:
blob = pickle.load(open(HERE / "data_small.pkl", "rb"))   # mixed subset (full curation = data_full.pkl, ~9GB)
train = [Item.model_validate(d) for d in blob["train"]]
test = [Item.model_validate(d) for d in blob["test"]]
SAMPLE = test[:50]
print(f"train={len(train):,}  test={len(test):,}")

train=16,801  test=2,000


## Zero-shot baseline (frontier reference)
`gpt-4o-mini`, no training. Hit = within 20% or $40 of the true price.

In [3]:
SYSTEM = "You estimate the price of products. Reply with just the price, e.g. 'Price is $9.00'."

def get_price(text):
    nums = re.findall(r"[-+]?\d*\.?\d+", text.replace(",", ""))   # first number in the text
    return float(nums[0]) if nums else 0.0

def score(errs, items):
    hits = sum(e <= 0.2 * it.price or e <= 40 for e, it in zip(errs, items))
    return sum(errs) / len(errs), hits / len(items)

def gpt_price(item):
    r = openai.chat.completions.create(model="gpt-4o-mini", seed=42, max_tokens=8,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": item.test_prompt()}])
    return get_price(r.choices[0].message.content)

mae0, hit0 = score([abs(gpt_price(it) - it.price) for it in SAMPLE], SAMPLE)
print(f"zero-shot gpt-4o-mini:  MAE=${mae0:,.2f}  hit-rate={hit0:.0%}")

zero-shot gpt-4o-mini:  MAE=$22.80  hit-rate=86%


## Fine-tune distilgpt2 with LoRA (local, CPU)
LoRA trains tiny adapter weights (~0.2% of params), so it's cheap. We train on the curated
prompts (each ends with `Price is $X.00`). Left-truncation keeps that price suffix.

In [4]:
tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.truncation_side = "left"                       # keep the END of the prompt (the price target)
model = get_peft_model(AutoModelForCausalLM.from_pretrained("distilgpt2"),
                       LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()

texts = [it.prompt for it in train[:150]]          # full training prompts
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
model.train()
for epoch in range(2):
    random.Random(epoch).shuffle(texts)
    for txt in texts:
        enc = tok(txt, return_tensors="pt", truncation=True, max_length=220)
        enc["labels"] = enc["input_ids"]
        model(**enc).loss.backward()
        opt.step(); opt.zero_grad()
    print(f"epoch {epoch + 1} done")

c:\Users\Nicholas Dean\projects\llm_engineering\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


epoch 1 done


epoch 2 done


## Compare: base distilgpt2 vs fine-tuned vs frontier

In [5]:
def local_price(item):
    enc = tok(item.test_prompt(), return_tensors="pt", truncation=True, max_length=220)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    return get_price(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))

def eval_local(items):
    return score([abs(local_price(it) - it.price) for it in items], items)

model.eval()
mae_ft, hit_ft = eval_local(SAMPLE)                      # adapters ON (fine-tuned)
with model.disable_adapter():
    mae_base, hit_base = eval_local(SAMPLE)              # adapters OFF (base distilgpt2)

print(f"base distilgpt2:       MAE=${mae_base:,.2f}  hit-rate={hit_base:.0%}")
print(f"fine-tuned distilgpt2: MAE=${mae_ft:,.2f}  hit-rate={hit_ft:.0%}")
print(f"gpt-4o-mini zero-shot: MAE=${mae0:,.2f}  hit-rate={hit0:.0%}  (frontier reference)")

base distilgpt2:       MAE=$97.36  hit-rate=70%
fine-tuned distilgpt2: MAE=$52.78  hit-rate=76%
gpt-4o-mini zero-shot: MAE=$22.80  hit-rate=86%  (frontier reference)
